# Fine-Tune a UNET Model for MNIST Digit Denoising

This notebook demonstrates how to fine-tune a UNET model for denoising MNIST digit images.

## Overview
- **Task**: Image denoising using UNET architecture
- **Dataset**: MNIST (with artificially added noise)
- **Architecture**: UNET - A convolutional neural network with encoder-decoder structure
- **Goal**: Remove noise from corrupted digit images

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Preparation

We'll load MNIST dataset and add noise to create training pairs of (noisy_image, clean_image).

In [ ]:
class NoisyMNIST(Dataset):
    """MNIST dataset with added Gaussian noise."""
    
    def __init__(self, root='./data', train=True, noise_factor=0.5):
        self.noise_factor = noise_factor
        
        # Load MNIST
        self.mnist = torchvision.datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )
    
    def __len__(self):
        return len(self.mnist)
    
    def __getitem__(self, idx):
        clean_img, label = self.mnist[idx]
        
        # Add Gaussian noise
        noise = torch.randn_like(clean_img) * self.noise_factor
        noisy_img = clean_img + noise
        
        # Clip to valid range [0, 1]
        noisy_img = torch.clamp(noisy_img, 0., 1.)
        
        return noisy_img, clean_img

# Create datasets
train_dataset = NoisyMNIST(train=True, noise_factor=0.5)
test_dataset = NoisyMNIST(train=False, noise_factor=0.5)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

### Visualize Noisy vs Clean Images

In [ ]:
# Display some examples
def show_images(noisy, clean, num_images=5):
    fig, axes = plt.subplots(2, num_images, figsize=(15, 6))
    
    for i in range(num_images):
        # Noisy images
        axes[0, i].imshow(noisy[i].squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Noisy', fontsize=12)
        
        # Clean images
        axes[1, i].imshow(clean[i].squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Clean', fontsize=12)
    
    plt.tight_layout()
    plt.show()

# Get a batch and visualize
noisy_batch, clean_batch = next(iter(train_loader))
show_images(noisy_batch, clean_batch)

## 3. UNET Model Architecture

UNET consists of:
- **Encoder (Contracting Path)**: Captures context through downsampling
- **Decoder (Expanding Path)**: Enables precise localization through upsampling
- **Skip Connections**: Concatenate encoder features to decoder for better gradient flow

In [ ]:
class DoubleConv(nn.Module):
    """Double convolution block: (Conv -> BatchNorm -> ReLU) * 2"""
    
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downsampling block: MaxPool -> DoubleConv"""
    
    def __init__(self, in_channels, out_channels):
        super(Down, self).__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upsampling block: Upsample -> DoubleConv"""
    
    def __init__(self, in_channels, out_channels):
        super(Up, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # Concatenate skip connection
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNET(nn.Module):
    """UNET architecture for image denoising."""
    
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super(UNET, self).__init__()
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()
        
        # Encoder (downsampling)
        self.encoder.append(DoubleConv(in_channels, features[0]))
        for i in range(len(features) - 1):
            self.encoder.append(Down(features[i], features[i + 1]))
        
        # Bottleneck
        self.bottleneck = Down(features[-1], features[-1] * 2)
        
        # Decoder (upsampling)
        for i in reversed(range(len(features))):
            if i == len(features) - 1:
                self.decoder.append(Up(features[i] * 2 * 2, features[i]))
            else:
                self.decoder.append(Up(features[i + 1], features[i]))
        
        # Final output layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        skip_connections = []
        
        # Encoder
        for i, encode in enumerate(self.encoder):
            x = encode(x)
            skip_connections.append(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder with skip connections
        skip_connections = skip_connections[::-1]
        for i, decode in enumerate(self.decoder):
            x = decode(x, skip_connections[i])
        
        return torch.sigmoid(self.final_conv(x))


# Initialize model
model = UNET(in_channels=1, out_channels=1).to(device)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

## 4. Training Configuration

In [ ]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# Training parameters
num_epochs = 10
best_loss = float('inf')

# Create directory for saving models
os.makedirs('./checkpoints', exist_ok=True)

## 5. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(dataloader, desc='Training')
    for noisy_imgs, clean_imgs in pbar:
        noisy_imgs = noisy_imgs.to(device)
        clean_imgs = clean_imgs.to(device)
        
        # Forward pass
        outputs = model(noisy_imgs)
        loss = criterion(outputs, clean_imgs)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    
    with torch.no_grad():
        for noisy_imgs, clean_imgs in tqdm(dataloader, desc='Validation'):
            noisy_imgs = noisy_imgs.to(device)
            clean_imgs = clean_imgs.to(device)
            
            outputs = model(noisy_imgs)
            loss = criterion(outputs, clean_imgs)
            running_loss += loss.item()
    
    return running_loss / len(dataloader)


# Training history
train_losses = []
val_losses = []

# Training loop
print("Starting training...\n")
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print("-" * 50)
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate(model, test_loader, criterion, device)
    val_losses.append(val_loss)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}\n")
    
    # Save best model
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, './checkpoints/best_unet_denoising.pth')
        print(f"✓ Saved best model (val_loss: {best_loss:.4f})\n")

print("Training completed!")

## 6. Plot Training History

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluate and Visualize Results

In [ ]:
# Load best model
checkpoint = torch.load('./checkpoints/best_unet_denoising.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
print(f"Best validation loss: {checkpoint['loss']:.4f}")

In [ ]:
def visualize_denoising(model, dataloader, device, num_images=8):
    """Visualize denoising results."""
    model.eval()
    
    # Get a batch
    noisy_imgs, clean_imgs = next(iter(dataloader))
    noisy_imgs = noisy_imgs.to(device)
    
    # Generate denoised images
    with torch.no_grad():
        denoised_imgs = model(noisy_imgs)
    
    # Move to CPU for visualization
    noisy_imgs = noisy_imgs.cpu()
    clean_imgs = clean_imgs.cpu()
    denoised_imgs = denoised_imgs.cpu()
    
    # Plot
    fig, axes = plt.subplots(3, num_images, figsize=(20, 8))
    
    for i in range(num_images):
        # Noisy
        axes[0, i].imshow(noisy_imgs[i].squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Noisy Input', fontsize=14, fontweight='bold')
        
        # Denoised
        axes[1, i].imshow(denoised_imgs[i].squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Denoised Output', fontsize=14, fontweight='bold')
        
        # Ground Truth
        axes[2, i].imshow(clean_imgs[i].squeeze(), cmap='gray')
        axes[2, i].axis('off')
        if i == 0:
            axes[2, i].set_title('Ground Truth', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize results
visualize_denoising(model, test_loader, device, num_images=8)

## 8. Quantitative Evaluation

In [ ]:
def calculate_psnr(img1, img2):
    """Calculate Peak Signal-to-Noise Ratio."""
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(1.0 / torch.sqrt(mse))


def evaluate_metrics(model, dataloader, device):
    """Evaluate model with PSNR and MSE metrics."""
    model.eval()
    total_mse = 0.0
    total_psnr = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for noisy_imgs, clean_imgs in tqdm(dataloader, desc='Evaluating'):
            noisy_imgs = noisy_imgs.to(device)
            clean_imgs = clean_imgs.to(device)
            
            denoised_imgs = model(noisy_imgs)
            
            # Calculate metrics
            mse = torch.mean((denoised_imgs - clean_imgs) ** 2).item()
            psnr = calculate_psnr(denoised_imgs, clean_imgs).item()
            
            total_mse += mse
            total_psnr += psnr
            num_batches += 1
    
    avg_mse = total_mse / num_batches
    avg_psnr = total_psnr / num_batches
    
    return avg_mse, avg_psnr


# Evaluate on test set
test_mse, test_psnr = evaluate_metrics(model, test_loader, device)

print("\n" + "="*50)
print("Test Set Evaluation Results")
print("="*50)
print(f"Mean Squared Error (MSE): {test_mse:.6f}")
print(f"Peak Signal-to-Noise Ratio (PSNR): {test_psnr:.2f} dB")
print("="*50)

## 9. Test on Custom Noise Levels

In [ ]:
# Test with different noise levels
noise_levels = [0.3, 0.5, 0.7]

fig, axes = plt.subplots(len(noise_levels), 8, figsize=(20, 3 * len(noise_levels)))

for row, noise_factor in enumerate(noise_levels):
    # Create dataset with specific noise level
    test_noise_dataset = NoisyMNIST(train=False, noise_factor=noise_factor)
    test_noise_loader = DataLoader(test_noise_dataset, batch_size=8, shuffle=True)
    
    # Get batch
    noisy_imgs, clean_imgs = next(iter(test_noise_loader))
    noisy_imgs_gpu = noisy_imgs.to(device)
    
    # Denoise
    with torch.no_grad():
        denoised_imgs = model(noisy_imgs_gpu).cpu()
    
    # Plot
    for col in range(8):
        # Show denoised result
        axes[row, col].imshow(denoised_imgs[col].squeeze(), cmap='gray')
        axes[row, col].axis('off')
        
        if col == 0:
            axes[row, col].set_ylabel(f'Noise: {noise_factor}', fontsize=12, fontweight='bold')

plt.suptitle('Denoising Results with Different Noise Levels', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Save and Export Model

In [ ]:
# Save final model
torch.save(model.state_dict(), './checkpoints/unet_denoising_final.pth')
print("Final model saved to './checkpoints/unet_denoising_final.pth'")

# Save entire model (architecture + weights)
torch.save(model, './checkpoints/unet_denoising_complete.pth')
print("Complete model saved to './checkpoints/unet_denoising_complete.pth'")

print("\nTo load the model later:")
print("model = torch.load('./checkpoints/unet_denoising_complete.pth')")
print("# or")
print("model = UNET()")
print("model.load_state_dict(torch.load('./checkpoints/unet_denoising_final.pth'))")

## Summary

In this notebook, we:

1. **Prepared the dataset**: Loaded MNIST and added Gaussian noise to create training pairs
2. **Built UNET architecture**: Implemented encoder-decoder structure with skip connections
3. **Trained the model**: Fine-tuned for digit denoising task
4. **Evaluated performance**: Used MSE and PSNR metrics
5. **Visualized results**: Compared noisy inputs, denoised outputs, and ground truth
6. **Tested generalization**: Evaluated on different noise levels

### Key Takeaways:
- UNET's skip connections help preserve spatial information during reconstruction
- The model learns to distinguish signal (digits) from noise (Gaussian)
- Performance can vary with different noise levels
- The architecture is flexible and can be adapted for other image-to-image tasks

### Next Steps:
- Experiment with different noise types (salt & pepper, speckle)
- Try data augmentation techniques
- Explore different loss functions (L1, perceptual loss)
- Apply to other datasets (CIFAR-10, custom images)